# 125 — Paralelismo, fan-out y map-reduce

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Fan-out**: dividir en n partes independientes y lanzar n workers simultáneos.
**Fan-in/reduce**: fusionar (deduplicar + contradicciones + síntesis; concatenar no es
fusionar). Herencia directa de MapReduce (Dean & Ghemawat, 2004).

**Coste a mano**: `coste = n·(t_in·p_in + t_out·p_out) + reduce`, donde el reduce lee
las n salidas → su entrada crece O(n) (con n grande: reduce jerárquico, ⌈log₂ n⌉
niveles). **Latencia** = `max(latencia_i)` + reduce: el rezagado manda.

**Ley del patrón**: el paralelo no abarata — compra latencia pagando tokens. Con
partes disjuntas el sobrecoste es ~0; con contexto compartido se acerca al ~15× que
reporta Anthropic.


## 🧮 Ejemplo de referencia

8 contratos, `p_in = 3 USD/MTok`, `p_out = 15 USD/MTok`; worker: 6 000 in / 500 out;
reduce: 400 de prompt + 8×500 de entrada, 1 200 out.

```text
map    = 8 × (0.0180 + 0.0075) = 0.2040 USD
reduce = 4 400×3e-6 + 1 200×15e-6 = 0.0132 + 0.0180 = 0.0312 USD
TOTAL  = 0.2352 USD   |   latencia ≈ 2 pasos vs 9 del secuencial
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("multiagent", seed=125)
show(result)


## Reflexión

1. En el ejemplo, map cuesta 0.204 USD y reduce 0.031 USD. ¿Con qué crecimiento de n (o de t_out por worker) el reduce pasa a dominar el coste y qué harías entonces?
2. Los tres workers del laboratorio corren lógicamente en paralelo porque no comparten estado. ¿Qué cambio en la tarea (no en el código) rompería esa independencia y obligaría a abandonar map-reduce?
3. Si duplicar n reduce la latencia a la mitad pero duplica el coste, ¿qué dato de negocio necesitas para elegir n? Formula la decisión como una desigualdad.
